In [1]:
using Gridap
using GridapGmsh

In [2]:
# Call Model
model_name = "2D_Plate_With_Hole.msh"
model_file = joinpath(@__DIR__,"..", "Model_Creation", "Quadileteral_Gmsh_Model", "Model", model_name)
isfile(model_file) || error("Path does not exist: $model_file")

true

In [3]:
model = GmshDiscreteModel(model_file)

Info    : Reading 'c:\Users\IIT BBSR\Desktop\Amiya\BTP\FEM_Problem\..\Model_Creation\Quadileteral_Gmsh_Model\Model\2D_Plate_With_Hole.msh'...
Info    : 18 entities
Info    : 751 nodes
Info    : 819 elements
Info    : Done reading 'c:\Users\IIT BBSR\Desktop\Amiya\BTP\FEM_Problem\..\Model_Creation\Quadileteral_Gmsh_Model\Model\2D_Plate_With_Hole.msh'


UnstructuredDiscreteModel()

In [4]:
const E = 210000.0                              # N/mm^2
const ν = 0.3                                   # Poissons Ratio

const μ = E/(2*(1+ν))
const λ1 = (E*ν)/((1+ν)*(1-2*ν))

# plane stress correction
const λ  = (2*λ1*μ)/(λ1+2*μ)  

69230.76923076923

In [5]:
g = VectorValue(100.0, 0.0)                     # Neumann Boundary Condition
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε                   # Sress Tensor

σ (generic function with 1 method)

In [6]:
# Γ_fixed   = tags = "D"
# Γ_Loaded = tags = "B"

In [ ]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V0 = TestFESpace(model,reffe;
    conformity=:H1,
    dirichlet_tags=["D"],
    dirichlet_masks=[(true,true)])
  
g1 = VectorValue(0.0, 0.0)                      # Drichlet Boundary Condition
U = TrialFESpace(V0,[g1])                           

TrialFESpace()

In [8]:
degree = 2*order
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)
Γ_load  = BoundaryTriangulation(model, tags = "B")
dΓ_N = Measure(Γ_load,degree)

GenericMeasure()

In [9]:
# Weak from
# Internal work (Stiffness component)
a(u,v) = ∫( ε(v) ⊙ (σ∘ε(u)) )*dΩ 
# External Work
l(v) = ∫(v⋅g)*dΓ_N

l (generic function with 1 method)

In [10]:
# Solve
op = AffineFEOperator(a,l,U,V0)
uh = solve(op)

SingleFieldFEFunction():
 num_cells: 683
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 12662350701452783345

In [11]:
# create vtu file for checkup
result_path = joinpath(@__DIR__, "..", "..", "Result", "FEM_Problems", "2D_Plate_With_Hole_FEM")
isdir(result_path) || mkpath(result_path)

"c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\FEM_Problem\\..\\..\\Result\\FEM_Problems\\2D_Plate_With_Hole_FEM"

In [12]:
writevtk(Ω,joinpath(result_path, "Output"),
    cellfields=[
        "Displacement"=>uh,
        "Strain"=>ε(uh),
        "Stress"=>σ∘ε(uh)]
        )

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\FEM_Problem\\..\\..\\Result\\FEM_Problems\\2D_Plate_With_Hole_FEM\\Output.vtu"],)